# 01C Retrieve Non-EU Documents

This workbook now follows the live non-EU retrieval workflow from `NID_Retrieval_Pipeline_non_EU_countries.ipynb` much more closely: editable terms, source-specific live retrieval, per-country inspection, live full-text extraction, cleaned outputs, and explicit per-source fallback only if enabled.

## Editable parameters in this notebook

Primary edit points:
- non-EU search terms
- source inclusion and country jobs
- live retrieval limits and pacing
- whether per-source fallback is allowed
- full-text extraction settings
- save paths and per-country CSV exports

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import os
import importlib

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "docs_df_cleaned").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

PIPE_DIR = ROOT / "analysis_pipeline"
PIPE_OUT = PIPE_DIR / "outputs"
PIPE_OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 60)
SAVE_OUTPUTS = True
MISSING_TEXT_VALUES = {"", "nan", "none", "null", "<na>"}


def preview_df(name: str, df: pd.DataFrame, n: int = 5) -> None:
    print(f"{name}: shape={df.shape}")
    display(df.head(n))


def write_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if SAVE_OUTPUTS:
        df.to_csv(path, index=index)
    print(f"{'Wrote' if SAVE_OUTPUTS else 'Would write'}: {path}")
    print(f"Shape: {df.shape}")
    display(df.head())


def text_series(series: pd.Series) -> pd.Series:
    return series.astype("string").fillna("").str.strip()


def missing_text_count(series: pd.Series) -> int:
    return int(text_series(series).str.lower().isin(MISSING_TEXT_VALUES).sum())


def fill_from_fulltext(base: pd.Series, fallback: pd.Series) -> pd.Series:
    base_text = text_series(base)
    return base.where(~base_text.str.lower().isin(MISSING_TEXT_VALUES), fallback)


import analysis_pipeline.functions.non_eu_retrieval as non_eu_retrieval_module
import analysis_pipeline.functions.retrieval_queries as retrieval_queries_module
importlib.reload(non_eu_retrieval_module)
importlib.reload(retrieval_queries_module)

from analysis_pipeline.functions.retrieval_queries import SEARCH_TERMS_PRIMARY
from analysis_pipeline.functions.non_eu_retrieval import (
    TRUSTSTORE_OK,
    fetch_uk_documents,
    fetch_aus_documents,
    fetch_us_documents,
    fetch_canada_documents,
    fetch_nz_documents,
    build_non_eu_doc_tables,
    build_non_eu_fulltext_docs,
    build_and_save_country_dfs,
    reconstruct_non_eu_hits_from_cache,
)

CANONICAL_ALL_DOCS = ROOT / "docs_df_cleaned" / "all_docs_df.csv"
OUT_RAW = PIPE_OUT / "01C_non_eu_raw_hits.csv"
OUT_DOCS = PIPE_OUT / "01C_non_eu_country_docs.csv"
OUT_FULLTEXT = PIPE_OUT / "01C_non_eu_fulltext_docs.csv"
COUNTRY_RAW_DIR = PIPE_OUT / "01C_country_raw"
COUNTRY_FULLTEXT_DIR = PIPE_OUT / "01C_country_fulltext"

os.environ["US_GOV_API_KEY"] = "10zafacZ8ALNpD8lErHcq30zJ1aqkBL6zGcfPOZ0"



## 1. Editable search terms

Keep these explicit in the notebook so you can edit them before running retrieval.

In [ ]:
SEARCH_TERMS = list(SEARCH_TERMS_PRIMARY)

INCLUDE_SOURCES = {
    "UK": True,
    "AUS": True,
    "US": True,
    "CA": True,
    "NZ": True,
}

RUN_LIVE = True
ALLOW_PER_SOURCE_FALLBACK = False
MAX_PER_TERM = 500
SLEEP_S = 0.25
NZ_MAX_PAGES = 5
FULLTEXT_MAX_WORKERS = 8
FULLTEXT_PROGRESS_EVERY = 25
FULLTEXT_OBEY_ROBOTS = True

US_API_KEY = os.getenv("US_GOV_API_KEY", "")

print("truststore injected =", TRUSTSTORE_OK)
print("US_API_KEY available =", bool(US_API_KEY))
display(pd.DataFrame({"term": SEARCH_TERMS}))
display(pd.DataFrame(list(INCLUDE_SOURCES.items()), columns=["source", "enabled"]))

truststore injected = False
US_API_KEY available = True


,term
0,nature-positive
1,nature positive
2,nature inclusive
3,nature-inclusive
4,nature based solutions
5,nature-based solutions
6,biodiversity net gain
7,biodiversity net-gain
8,biodiversity gain
9,nature restoration


,source,enabled
0,UK,True
1,AUS,True
2,US,True
3,CA,True
4,NZ,True


## 2. Country/source job setup

This preserves the source-specific nature of the original notebook rather than hiding everything behind a fake generic wrapper.

In [6]:
source_jobs = [
    {
        "source": "UK",
        "country": "United Kingdom",
        "jurisdiction": "United Kingdom",
        "runner": fetch_uk_documents,
        "runner_name": "fetch_uk_documents",
        "kwargs": {"max_per_term": MAX_PER_TERM, "sleep_s": SLEEP_S},
    },
    {
        "source": "AUS",
        "country": "Australia",
        "jurisdiction": "Australia",
        "runner": fetch_aus_documents,
        "runner_name": "fetch_aus_documents",
        "kwargs": {"max_per_term": MAX_PER_TERM, "sleep_s": SLEEP_S},
    },
    {
        "source": "US",
        "country": "United States",
        "jurisdiction": "United States",
        "runner": fetch_us_documents,
        "runner_name": "fetch_us_documents",
        "kwargs": {"api_key": US_API_KEY, "max_per_term": MAX_PER_TERM, "sleep_s": SLEEP_S},
    },
    {
        "source": "CA",
        "country": "Canada",
        "jurisdiction": "Canada",
        "runner": fetch_canada_documents,
        "runner_name": "fetch_canada_documents",
        "kwargs": {"max_per_term": MAX_PER_TERM, "sleep_s": SLEEP_S},
    },
    {
        "source": "NZ",
        "country": "New Zealand",
        "jurisdiction": "New Zealand",
        "runner": fetch_nz_documents,
        "runner_name": "fetch_nz_documents",
        "kwargs": {"max_per_term": MAX_PER_TERM, "sleep_s": SLEEP_S, "max_pages": NZ_MAX_PAGES},
    },
]

job_df = pd.DataFrame(
    [
        {
            "source": job["source"],
            "country": job["country"],
            "enabled": INCLUDE_SOURCES.get(job["source"], False),
            "runner": job["runner_name"],
            "term_count": len(SEARCH_TERMS),
            "fallback_allowed": ALLOW_PER_SOURCE_FALLBACK,
        }
        for job in source_jobs
    ]
)
preview_df("job_df", job_df, n=len(job_df))

job_df: shape=(5, 6)


,source,country,enabled,runner,term_count,fallback_allowed
0,UK,United Kingdom,True,fetch_uk_documents,12,False
1,AUS,Australia,True,fetch_aus_documents,12,False
2,US,United States,True,fetch_us_documents,12,False
3,CA,Canada,True,fetch_canada_documents,12,False
4,NZ,New Zealand,True,fetch_nz_documents,12,False


## 3. Run source retrieval for each country

The main path is live retrieval. Fallback, if enabled, happens only after an attempted source-specific live run and is recorded per source.

In [7]:
empty_raw = build_non_eu_doc_tables(pd.DataFrame())[0]
empty_full = pd.DataFrame(columns=["doc_id", "jurisdiction", "doc_uid", "title", "url", "lang", "source_file", "full_text_clean", "text_len", "has_text"])

country_raw_hits = {}
country_fallback_fulltext = {}
retrieval_runs = []
nz_retrieval_diag_df = pd.DataFrame()
enabled_jobs = [job for job in source_jobs if INCLUDE_SOURCES.get(job["source"], False)]

print(f"Running source retrieval for {len(enabled_jobs)} country jobs")

for job_index, job in enumerate(enabled_jobs, start=1):
    source = job["source"]
    country = job["country"]
    jurisdiction = job["jurisdiction"]

    print()
    print(f"[{job_index}/{len(enabled_jobs)}] Starting {country} ({source})")
    print(f"runner = {job['runner_name']} | terms = {len(SEARCH_TERMS)} | run_live = {RUN_LIVE} | fallback = {ALLOW_PER_SOURCE_FALLBACK}")

    mode = "not_run"
    live_error = ""
    live_rows = 0
    final_rows = 0
    unique_docs = 0
    raw_df = empty_raw.copy()
    fallback_full_df = empty_full.copy()

    if RUN_LIVE:
        try:
            if source == "NZ":
                raw_result = job["runner"](SEARCH_TERMS, **job["kwargs"], verbose=True, return_diagnostics=True)
                raw_df, nz_retrieval_diag_df = raw_result
                if not nz_retrieval_diag_df.empty:
                    print("[NZ] diagnostics preview")
                    display(nz_retrieval_diag_df.head(10))
            else:
                raw_df = job["runner"](SEARCH_TERMS, **job["kwargs"])
            live_rows = len(raw_df)
            unique_docs = raw_df["doc_id"].nunique() if not raw_df.empty and "doc_id" in raw_df.columns else 0
            print(f"[{job_index}/{len(enabled_jobs)}] Live retrieval returned {live_rows} rows / {unique_docs} unique docs for {country}")
            if live_rows > 0:
                mode = "live"
            elif ALLOW_PER_SOURCE_FALLBACK:
                print(f"[{job_index}/{len(enabled_jobs)}] Live retrieval was empty for {country}; switching to explicit fallback")
                raw_df, fallback_full_df = reconstruct_non_eu_hits_from_cache(CANONICAL_ALL_DOCS, SEARCH_TERMS, jurisdiction=jurisdiction)
                mode = "fallback_after_empty_live"
            else:
                mode = "live_empty"
        except Exception as exc:
            live_error = str(exc)
            print(f"[{job_index}/{len(enabled_jobs)}] Live retrieval failed for {country}: {live_error}")
            if ALLOW_PER_SOURCE_FALLBACK:
                print(f"[{job_index}/{len(enabled_jobs)}] Switching to explicit fallback for {country}")
                raw_df, fallback_full_df = reconstruct_non_eu_hits_from_cache(CANONICAL_ALL_DOCS, SEARCH_TERMS, jurisdiction=jurisdiction)
                mode = "fallback_after_live_error"
            else:
                mode = "live_failed_no_fallback"
    elif ALLOW_PER_SOURCE_FALLBACK:
        print(f"[{job_index}/{len(enabled_jobs)}] Live retrieval disabled; using explicit fallback for {country}")
        raw_df, fallback_full_df = reconstruct_non_eu_hits_from_cache(CANONICAL_ALL_DOCS, SEARCH_TERMS, jurisdiction=jurisdiction)
        mode = "fallback_only"

    country_raw_hits[country] = raw_df.copy()
    country_fallback_fulltext[country] = fallback_full_df.copy()
    final_rows = len(raw_df)
    unique_docs = raw_df["doc_id"].nunique() if not raw_df.empty and "doc_id" in raw_df.columns else 0
    retrieval_runs.append(
        {
            "source": source,
            "country": country,
            "mode": mode,
            "live_rows": live_rows,
            "final_rows": final_rows,
            "unique_docs": unique_docs,
            "live_error": live_error,
        }
    )
    print(f"[{job_index}/{len(enabled_jobs)}] Completed {country} | mode = {mode} | final_rows = {final_rows} | unique_docs = {unique_docs}")

retrieval_runs_df = pd.DataFrame(retrieval_runs)
preview_df("retrieval_runs_df", retrieval_runs_df, n=len(retrieval_runs_df))


Running source retrieval for 5 country jobs

[1/5] Starting United Kingdom (UK)
runner = fetch_uk_documents | terms = 12 | run_live = True | fallback = False
[1/5] Live retrieval returned 142 rows / 85 unique docs for United Kingdom
[1/5] Completed United Kingdom | mode = live | final_rows = 142 | unique_docs = 85

[2/5] Starting Australia (AUS)
runner = fetch_aus_documents | terms = 12 | run_live = True | fallback = False
[2/5] Live retrieval returned 46 rows / 40 unique docs for Australia
[2/5] Completed Australia | mode = live | final_rows = 46 | unique_docs = 40

[3/5] Starting United States (US)
runner = fetch_us_documents | terms = 12 | run_live = True | fallback = False
[3/5] Live retrieval returned 886 rows / 837 unique docs for United States
[3/5] Completed United States | mode = live | final_rows = 886 | unique_docs = 837

[4/5] Starting Canada (CA)
runner = fetch_canada_documents | terms = 12 | run_live = True | fallback = False
[4/5] Live retrieval returned 49 rows / 32 uni

,host,term,page,status_code,candidates_found,new_urls_kept,kept_total,stop_reason,request_url
0,www.legislation.govt.nz,nature-positive,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature-positive""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D=&..."
1,www.legislation.govt.nz,nature positive,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature+positive""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D=&..."
2,www.legislation.govt.nz,nature inclusive,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature+inclusive""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D=..."
3,www.legislation.govt.nz,nature-inclusive,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature-inclusive""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D=..."
4,www.legislation.govt.nz,nature based solutions,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature+based+solutions""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5..."
5,www.legislation.govt.nz,nature-based solutions,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature-based+solutions""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5..."
6,www.legislation.govt.nz,biodiversity net gain,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""biodiversity+net+gain""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5B..."
7,www.legislation.govt.nz,biodiversity net-gain,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""biodiversity+net-gain""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5B..."
8,www.legislation.govt.nz,biodiversity gain,1,200,2,2,2,continue,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""biodiversity+gain""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D..."
9,www.legislation.govt.nz,biodiversity gain,2,200,0,0,2,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""biodiversity+gain""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D..."


[5/5] Live retrieval returned 7 rows / 7 unique docs for New Zealand
[5/5] Completed New Zealand | mode = live | final_rows = 7 | unique_docs = 7
retrieval_runs_df: shape=(5, 7)


,source,country,mode,live_rows,final_rows,unique_docs,live_error
0,UK,United Kingdom,live,142,142,85,
1,AUS,Australia,live,46,46,40,
2,US,United States,live,886,886,837,
3,CA,Canada,live,49,49,32,
4,NZ,New Zealand,live,7,7,7,


## 4. Inspect raw hits per country

In [8]:
raw_hit_summary = pd.DataFrame(
    [
        {
            "country": country,
            "rows": len(df),
            "documents": df["doc_id"].nunique() if not df.empty and "doc_id" in df.columns else 0,
            "matched_terms": df["term"].dropna().nunique() if not df.empty and "term" in df.columns else 0,
            "missing_title": missing_text_count(df["title"]) if not df.empty and "title" in df.columns else 0,
            "missing_year": int(df["year"].isna().sum()) if not df.empty and "year" in df.columns else 0,
        }
        for country, df in country_raw_hits.items()
    ]
)
preview_df("raw_hit_summary", raw_hit_summary, n=len(raw_hit_summary))

if not nz_retrieval_diag_df.empty:
    print("NZ term/page diagnostics")
    preview_df("nz_retrieval_diag_df", nz_retrieval_diag_df, n=20)
    nz_term_page_summary = (
        nz_retrieval_diag_df.groupby(["term", "page"], as_index=False)
        .agg(
            status_code=("status_code", "first"),
            candidates_found=("candidates_found", "sum"),
            new_urls_kept=("new_urls_kept", "sum"),
            kept_total=("kept_total", "max"),
            stop_reason=("stop_reason", "last"),
        )
    )
    preview_df("nz_term_page_summary", nz_term_page_summary, n=len(nz_term_page_summary))

countries_to_preview = list(country_raw_hits.items())
print(f"Inspecting raw hits for {len(countries_to_preview)} countries")
for preview_index, (country, df) in enumerate(countries_to_preview, start=1):
    print()
    print(f"[{preview_index}/{len(countries_to_preview)}] Inspecting raw hits for {country} | rows = {len(df)}")
    cols = [c for c in ["doc_id", "title", "year", "country", "source", "url", "term"] if c in df.columns]
    preview_df(country, df[cols] if cols else df, n=10)



raw_hit_summary: shape=(5, 4)


,country,rows,documents,matched_terms
0,United Kingdom,142,85,8
1,Australia,46,40,5
2,United States,886,837,10
3,Canada,49,32,6
4,New Zealand,7,7,2


NZ term/page diagnostics
nz_retrieval_diag_df: shape=(14, 9)


,host,term,page,status_code,candidates_found,new_urls_kept,kept_total,stop_reason,request_url
0,www.legislation.govt.nz,nature-positive,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature-positive""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D=&..."
1,www.legislation.govt.nz,nature positive,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature+positive""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D=&..."
2,www.legislation.govt.nz,nature inclusive,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature+inclusive""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D=..."
3,www.legislation.govt.nz,nature-inclusive,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature-inclusive""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D=..."
4,www.legislation.govt.nz,nature based solutions,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature+based+solutions""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5..."
5,www.legislation.govt.nz,nature-based solutions,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""nature-based+solutions""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5..."
6,www.legislation.govt.nz,biodiversity net gain,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""biodiversity+net+gain""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5B..."
7,www.legislation.govt.nz,biodiversity net-gain,1,200,0,0,0,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""biodiversity+net-gain""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5B..."
8,www.legislation.govt.nz,biodiversity gain,1,200,2,2,2,continue,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""biodiversity+gain""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D..."
9,www.legislation.govt.nz,biodiversity gain,2,200,0,0,2,no_candidates,"https://www.legislation.govt.nz/items/?search_field=content&search_term=""biodiversity+gain""&as%5Bty%5D%5B%5D=act&as%5Bty%5D%5B%5D=secondary_legislation&as%5Bty%5D%5B%5D=bill&as%5Bty%5D%5B%5D=amendment_paper&as%5Bt%5D..."


nz_term_page_summary: shape=(14, 7)


...

,jurisdiction,source,matched_term,term,api_id,api_self,doc_url,url,title,document_id,lang,doc_id,doc_uid
0,United States,US,nature-positive,nature-positive,EPA-R09-OAR-2023-0263-0434,https://api.regulations.gov/v4/documents/EPA-R09-OAR-2023-0263-0434,https://api.regulations.gov/v4/documents/EPA-R09-OAR-2023-0263-0434,https://api.regulations.gov/v4/documents/EPA-R09-OAR-2023-0263-0434,E-2_EarthJustice Comment 2,EPA-R09-OAR-2023-0263-0434,en,US:EPA-R09-OAR-2023-0263-0434,https://api.regulations.gov/v4/documents/EPA-R09-OAR-2023-0263-0434
1,United States,US,nature-positive,nature-positive,EPA-HQ-OAR-2022-0606-0162,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0162,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0162,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0162,Documents and Materials Referenced in the Preamble: Folder One,EPA-HQ-OAR-2022-0606-0162,en,US:EPA-HQ-OAR-2022-0606-0162,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0162
2,United States,US,nature-positive,nature-positive,EPA-HQ-OAR-2022-0606-0019,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0019,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0019,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0019,Technical Support Document: Cold Chain,EPA-HQ-OAR-2022-0606-0019,en,US:EPA-HQ-OAR-2022-0606-0019,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0019
3,United States,US,nature positive,nature positive,EPA-HQ-OAR-2022-0606-0162,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0162,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0162,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0162,Documents and Materials Referenced in the Preamble: Folder One,EPA-HQ-OAR-2022-0606-0162,en,US:EPA-HQ-OAR-2022-0606-0162,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2022-0606-0162
4,United States,US,nature positive,nature positive,APHIS-2020-0081-0029,https://api.regulations.gov/v4/documents/APHIS-2020-0081-0029,https://api.regulations.gov/v4/documents/APHIS-2020-0081-0029,https://api.regulations.gov/v4/documents/APHIS-2020-0081-0029,FEIS Appendix C: WDM Methods,APHIS-2020-0081-0029,en,US:APHIS-2020-0081-0029,https://api.regulations.gov/v4/documents/APHIS-2020-0081-0029
5,United States,US,nature positive,nature positive,APHIS-2020-0081-0014,https://api.regulations.gov/v4/documents/APHIS-2020-0081-0014,https://api.regulations.gov/v4/documents/APHIS-2020-0081-0014,https://api.regulations.gov/v4/documents/APHIS-2020-0081-0014,App C_WDM Methods,APHIS-2020-0081-0014,en,US:APHIS-2020-0081-0014,https://api.regulations.gov/v4/documents/APHIS-2020-0081-0014
6,United States,US,nature positive,nature positive,EPA-HQ-OAR-2021-0317-3964,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2021-0317-3964,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2021-0317-3964,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2021-0317-3964,EPA; Cited References (Group #13) for the Report on the Social Cost of Greenhouse Gases,EPA-HQ-OAR-2021-0317-3964,en,US:EPA-HQ-OAR-2021-0317-3964,https://api.regulations.gov/v4/documents/EPA-HQ-OAR-2021-0317-3964
7,United States,US,nature positive,nature positive,FAA-2023-1377-0002,https://api.regulations.gov/v4/documents/FAA-2023-1377-0002,https://api.regulations.gov/v4/documents/FAA-2023-1377-0002,https://api.regulations.gov/v4/documents/FAA-2023-1377-0002,U.S. DOT/FAA - Supplemental Documents,FAA-2023-1377-0002,en,US:FAA-2023-1377-0002,https://api.regulations.gov/v4/documents/FAA-2023-1377-0002
8,United States,US,nature positive,nature positive,APHIS-2022-0002-0002,https://api.regulations.gov/v4/documents/APHIS-2022-0002-0002,https://api.regulations.gov/v4/documents/APHIS-2022-0002-0002,https://api.regulations.gov/v4/documents/APHIS-2022-0002-0002,"Appendices: Hart Mountain National Antelope Refuge: Final Bighorn Sheep Management Plan and Environmental Impact Statement, November 2021",APHIS-2022-0002-0002


[4/5] Inspecting raw hits for Canada | rows = 49
Canada: shape=(49, 10)


,jurisdiction,source,matched_term,term,doc_url,url,title,lang,doc_id,doc_uid
0,Canada,CA,nature-positive,nature-positive,https://www.publications.gc.ca/site/eng/9.506446/publication.html,https://www.publications.gc.ca/site/eng/9.506446/publication.html,Annual report / Canadian Museum of Nature. NM10E-PDF,en,CA:https://www.publications.gc.ca/site/eng/9.506446/publication.html,https://www.publications.gc.ca/site/eng/9.506446/publication.html
1,Canada,CA,nature-positive,nature-positive,https://www.publications.gc.ca/site/eng/9.502842/publication.html,https://www.publications.gc.ca/site/eng/9.502842/publication.html,Rapport annuel de ... / Musée canadien de la nature. NM10F-PDF,en,CA:https://www.publications.gc.ca/site/eng/9.502842/publication.html,https://www.publications.gc.ca/site/eng/9.502842/publication.html
2,Canada,CA,nature-positive,nature-positive,https://www.publications.gc.ca/site/eng/9.930055/publication.html,https://www.publications.gc.ca/site/eng/9.930055/publication.html,Quarterly financial report (unaudited) for the ... month period ended ... / Canadian Museum of Nature. NM12-1E-PDF,en,CA:https://www.publications.gc.ca/site/eng/9.930055/publication.html,https://www.publications.gc.ca/site/eng/9.930055/publication.html
3,Canada,CA,nature-positive,nature-positive,https://www.publications.gc.ca/site/eng/9.930054/publication.html,https://www.publications.gc.ca/site/eng/9.930054/publication.html,Rapport financier trimestriel (non vérifié) pour la période de ... mois terminée le ... / Musée canadien de la nature. NM12-1F-PDF,en,CA:https://www.publications.gc.ca/site/eng/9.930054/publication.html,https://www.publications.gc.ca/site/eng/9.930054/publication.html
4,Canada,CA,nature-positive,nature-positive,https://www.publications.gc.ca/site/eng/9.899513/publication.html,https://www.publications.gc.ca/site/eng/9.899513/publication.html,"Résumé de l'étude du Programme d'action positive / préparé par le Groupe d'étude du Programme d'action positive, Direction du personnel. RG15-28/1985F-PDF",en,CA:https://www.publications.gc.ca/site/eng/9.899513/publication.html,https://www.publications.gc.ca/site/eng/9.899513/publication.html
5,Canada,CA,nature-positive,nature-positive,https://www.publications.gc.ca/site/eng/9.840878/publication.html,https://www.publications.gc.ca/site/eng/9.840878/publication.html,Year in review ... / Canadian Museum of Nature. NM10-1E-PDF,en,CA:https://www.publications.gc.ca/site/eng/9.840878/publication.html,https://www.publications.gc.ca/site/eng/9.840878/publication.html
6,Canada,CA,nature-positive,nature-positive,https://www.publications.gc.ca/site/eng/9.840883/publication.html,https://www.publications.gc.ca/site/eng/9.840883/publication.html,Rétrospective de ... / Musée canadien de la nature. NM10-1F-PDF,en,CA:https://www.publications.gc.ca/site/eng/9.840883/publication.html,https://www.publications.gc.ca/site/eng/9.840883/publication.html
7,Canada,CA,nature-positive,nature-positive,https://www.publications.gc.ca/site/eng/9.835175/publication.html,https://www.publications.gc.ca/site/eng/9.835175/publication.html,"Corporate plan ... summary, capital and operating budget / Canadian Museum of Nature. NM11-2E-PDF",en,CA:https://www.publications.gc.ca/site/eng/9.835175/publication.html,https://www.publications.gc.ca/site/eng/9.835175/publication.html
8,Canada,CA,nature-positive,nature-positive,https://www.publications.gc.ca/site/eng/9.689111/publication.html,https://www.publications.gc.ca/site/eng/9.689111/publication.html,Evaluation of the Family Violence Initiative - Multiculturalism Program : final report / Submitted by: Positive Outcomes Consulting Services. CH34-11-2002E-PDF,en,CA:https://www.publications.gc.ca/site/eng/9.689111/publication.html,https://www.publications.gc.ca/site/eng/9.689111/publication.html
9,Canada,CA,nature-positive,nature-positive,https://www.publications.gc.ca/site/eng/9.908402/publication.html,https://www.publications.gc.ca/site/eng/9.908402/publication.html,"Canadian Museum


[5/5] Inspecting raw hits for New Zealand | rows = 7
New Zealand: shape=(7, 10)


,jurisdiction,source,matched_term,term,doc_url,url,title,lang,doc_id,doc_uid
0,New Zealand,NZ,biodiversity gain,biodiversity gain,https://www.legislation.govt.nz/bill/government/2022/186/en/latest/highlights/?highlight=%22biodiversity+gain%22,https://www.legislation.govt.nz/bill/government/2022/186/en/latest/highlights/?highlight=%22biodiversity+gain%22,,en,NZ:https://www.legislation.govt.nz/bill/government/2022/186/en/latest/highlights,https://www.legislation.govt.nz/bill/government/2022/186/en/latest/highlights/?highlight=%22biodiversity+gain%22
1,New Zealand,NZ,biodiversity gain,biodiversity gain,https://www.legislation.govt.nz/act/public/2023/46/en/latest/highlights/?highlight=%22biodiversity+gain%22,https://www.legislation.govt.nz/act/public/2023/46/en/latest/highlights/?highlight=%22biodiversity+gain%22,,en,NZ:https://www.legislation.govt.nz/act/public/2023/46/en/latest/highlights,https://www.legislation.govt.nz/act/public/2023/46/en/latest/highlights/?highlight=%22biodiversity+gain%22
2,New Zealand,NZ,biodiversity strategy,biodiversity strategy,https://www.legislation.govt.nz/bill/members/2022/156/en/latest/highlights/?highlight=%22biodiversity+strategy%22,https://www.legislation.govt.nz/bill/members/2022/156/en/latest/highlights/?highlight=%22biodiversity+strategy%22,,en,NZ:https://www.legislation.govt.nz/bill/members/2022/156/en/latest/highlights,https://www.legislation.govt.nz/bill/members/2022/156/en/latest/highlights/?highlight=%22biodiversity+strategy%22
3,New Zealand,NZ,biodiversity strategy,biodiversity strategy,https://www.legislation.govt.nz/act/public/2016/20/en/latest/highlights/?highlight=%22biodiversity+strategy%22,https://www.legislation.govt.nz/act/public/2016/20/en/latest/highlights/?highlight=%22biodiversity+strategy%22,,en,NZ:https://www.legislation.govt.nz/act/public/2016/20/en/latest/highlights,https://www.legislation.govt.nz/act/public/2016/20/en/latest/highlights/?highlight=%22biodiversity+strategy%22
4,New Zealand,NZ,biodiversity strategy,biodiversity strategy,https://www.legislation.govt.nz/act/public/2010/12/en/latest/highlights/?highlight=%22biodiversity+strategy%22,https://www.legislation.govt.nz/act/public/2010/12/en/latest/highlights/?highlight=%22biodiversity+strategy%22,,en,NZ:https://www.legislation.govt.nz/act/public/2010/12/en/latest/highlights,https://www.legislation.govt.nz/act/public/2010/12/en/latest/highlights/?highlight=%22biodiversity+strategy%22
5,New Zealand,NZ,biodiversity strategy,biodiversity strategy,https://www.legislation.govt.nz/bill/government/2015/60/en/latest/highlights/?highlight=%22biodiversity+strategy%22,https://www.legislation.govt.nz/bill/government/2015/60/en/latest/highlights/?highlight=%22biodiversity+strategy%22,,en,NZ:https://www.legislation.govt.nz/bill/government/2015/60/en/latest/highlights,https://www.legislation.govt.nz/bill/government/2015/60/en/latest/highlights/?highlight=%22biodiversity+strategy%22
6,New Zealand,NZ,biodiversity strategy,biodiversity strategy,https://www.legislation.govt.nz/bill/government/2010/130/en/latest/highlights/?highlight=%22biodiversity+strategy%22,https://www.legislation.govt.nz/bill/government/2010/130/en/latest/highlights/?highlight=%22biodiversity+strategy%22,,en,NZ:https://www.legislation.govt.nz/bill/government/2010/130/en/latest/highlights,https://www.legislation.govt.nz/bill/government/2010/130/en/latest/highlights/?highlight=%22biodiversity+strategy%22


## 5. Run full-text extraction

Live full-text extraction is attempted for live retrieval outputs. If a source is already on explicit fallback, that cached fallback full-text table is carried forward visibly.

In [9]:
country_fulltext_docs = {}
fulltext_runs = []

for run in retrieval_runs:
    country = run["country"]
    mode = run["mode"]
    raw_df = country_raw_hits.get(country, empty_raw)
    fallback_full_df = country_fallback_fulltext.get(country, empty_full)
    fulltext_mode = "not_run"
    fulltext_error = ""

    if mode.startswith("fallback") and not fallback_full_df.empty:
        full_df = fallback_full_df.copy()
        fulltext_mode = "fallback_fulltext"
    else:
        try:
            full_df = build_non_eu_fulltext_docs(
                raw_df,
                us_api_key=US_API_KEY,
                max_workers=FULLTEXT_MAX_WORKERS,
                progress_every=FULLTEXT_PROGRESS_EVERY,
                obey_robots=FULLTEXT_OBEY_ROBOTS,
            )
            fulltext_mode = "live_fulltext"
            if full_df.empty and ALLOW_PER_SOURCE_FALLBACK:
                _, full_df = reconstruct_non_eu_hits_from_cache(CANONICAL_ALL_DOCS, SEARCH_TERMS, jurisdiction=country)
                fulltext_mode = "fallback_after_empty_fulltext"
        except Exception as exc:
            fulltext_error = str(exc)
            if ALLOW_PER_SOURCE_FALLBACK:
                _, full_df = reconstruct_non_eu_hits_from_cache(CANONICAL_ALL_DOCS, SEARCH_TERMS, jurisdiction=country)
                fulltext_mode = "fallback_after_fulltext_error"
            else:
                full_df = empty_full.copy()
                fulltext_mode = "fulltext_failed_no_fallback"

    country_fulltext_docs[country] = full_df.copy()
    fulltext_runs.append(
        {
            "country": country,
            "fulltext_mode": fulltext_mode,
            "rows": len(full_df),
            "docs_with_text": int(full_df["has_text"].sum()) if not full_df.empty and "has_text" in full_df.columns else 0,
            "fulltext_error": fulltext_error,
        }
    )

fulltext_runs_df = pd.DataFrame(fulltext_runs)
preview_df("fulltext_runs_df", fulltext_runs_df, n=len(fulltext_runs_df))

[PROGRESS] 25/85 | ok=25 | errors=0
[PROGRESS] 50/85 | ok=50 | errors=0
[PROGRESS] 75/85 | ok=75 | errors=0
[PROGRESS] 85/85 | ok=85 | errors=0
[PROGRESS] 25/40 | ok=25 | errors=0
[PROGRESS] 40/40 | ok=40 | errors=0
[PROGRESS] 25/837 | ok=25 | errors=0
[PROGRESS] 50/837 | ok=50 | errors=0
[PROGRESS] 75/837 | ok=75 | errors=0
[PROGRESS] 100/837 | ok=100 | errors=0
[PROGRESS] 125/837 | ok=125 | errors=0
[PROGRESS] 150/837 | ok=150 | errors=0
[PROGRESS] 175/837 | ok=175 | errors=0
[PROGRESS] 200/837 | ok=200 | errors=0
[PROGRESS] 225/837 | ok=225 | errors=0
[PROGRESS] 250/837 | ok=250 | errors=0
[PROGRESS] 275/837 | ok=275 | errors=0
[PROGRESS] 300/837 | ok=300 | errors=0
[PROGRESS] 325/837 | ok=325 | errors=0
[PROGRESS] 350/837 | ok=350 | errors=0
[PROGRESS] 375/837 | ok=375 | errors=0
[PROGRESS] 400/837 | ok=400 | errors=0
[PROGRESS] 425/837 | ok=425 | errors=0
[PROGRESS] 450/837 | ok=450 | errors=0
[PROGRESS] 475/837 | ok=475 | errors=0
[PROGRESS] 500/837 | ok=500 | errors=0
[PROGRESS]

,country,fulltext_mode,rows,docs_with_text,fulltext_error
0,United Kingdom,live_fulltext,85,85,
1,Australia,live_fulltext,40,40,
2,United States,live_fulltext,837,836,
3,Canada,live_fulltext,32,32,
4,New Zealand,live_fulltext,7,7,


## 6. Inspect full-text coverage per country

In [10]:
coverage_rows = []
for country, df in country_fulltext_docs.items():
    coverage_rows.append(
        {
            "country": country,
            "documents": len(df),
            "with_text": int(df["has_text"].sum()) if not df.empty and "has_text" in df.columns else 0,
            "mean_text_len": float(df["text_len"].mean()) if not df.empty and "text_len" in df.columns else 0.0,
            "missing_title": missing_text_count(df["title"]) if not df.empty and "title" in df.columns else 0,
            "missing_year": int(df["year"].isna().sum()) if not df.empty and "year" in df.columns else 0,
        }
    )
coverage_df = pd.DataFrame(coverage_rows)
if not coverage_df.empty:
    coverage_df["text_coverage"] = coverage_df["with_text"] / coverage_df["documents"].clip(lower=1)
preview_df("coverage_df", coverage_df, n=len(coverage_df))
for country, df in country_fulltext_docs.items():
    print()
    print(f"--- {country} full text ---")
    cols = [c for c in ["doc_id", "title", "year", "country", "url", "text_len", "has_text", "retrieval_status", "full_text_error"] if c in df.columns]
    preview_df(country, df[cols] if cols else df, n=10)



coverage_df: shape=(5, 5)


,country,documents,with_text,mean_text_len,text_coverage
0,United Kingdom,85,85,16434.517647,1.000000
1,Australia,40,40,5595.575000,1.000000
2,United States,837,836,194.385902,0.998805
3,Canada,32,32,3423.812500,1.000000
4,New Zealand,7,7,653393.285714,1.000000



--- United Kingdom full text ---
United Kingdom: shape=(85, 5)


,doc_id,title,text_len,has_text,full_text_error
0,UK:https://www.legislation.gov.uk/asp/2004/6/contents,,16015,True,
1,UK:https://www.legislation.gov.uk/asp/2018/8/contents,,11498,True,
2,UK:https://www.legislation.gov.uk/asp/2021/8/contents,,6529,True,
3,UK:https://www.legislation.gov.uk/asp/2022/3/contents,,6529,True,
4,UK:https://www.legislation.gov.uk/asp/2023/2/contents,,6529,True,
5,UK:https://www.legislation.gov.uk/asp/2024/11/contents,,9495,True,
6,UK:https://www.legislation.gov.uk/asp/2024/3/contents,,6529,True,
7,UK:https://www.legislation.gov.uk/asp/2025/7/contents,,6769,True,
8,UK:https://www.legislation.gov.uk/asp/2026/6/contents,,9167,True,
9,UK:https://www.legislation.gov.uk/nisr/2011/285/contents,,5411,True,



--- Australia full text ---
Australia: shape=(40, 5)


...

,doc_id,title,text_len,has_text,full_text_error
0,US:APHIS-2015-0068-0001,Environmental Assessment: Managing Blackbird Damage to Sprouting Rice in Southwestern Louisiana,198,True,
1,US:APHIS-2015-0068-0013,Final Environmental Assessment: Managing Blackbird Damage to Sprouting Rice in SW LA,186,True,
2,US:APHIS-2016-0003-0001,Environmental Assessment: Managing Damage to Resources and Threats to Human Safety Caused by Birds in the State of Alabama,225,True,
3,US:APHIS-2016-0003-0005,Final Environmental Assessment: Managing Damage to Resources and Threats to Human Safety Caused By Birds in the State Of Alabama,231,True,
4,US:APHIS-2016-0004-0001,Environmental Assessment: Cormorant and Gull Damage Management on Lake Champlain,183,True,
5,US:APHIS-2017-0023-0040,"Revised Environmental Assessment: Management of Wolf Conflicts and Depredating Wolves in Minnesota, November 2022",215,True,
6,US:APHIS-2017-0023-0047,"Final Environmental Assessment: Management of Wolf Conflicts and Depredating Wolves in Minnesota, February 2023",213,True,
7,US:APHIS-2017-0050-0001,Reducing Bird Damage in the State of South Carolina: Draft Environmental Assessment (Pre-Decisional),227,True,
8,US:APHIS-2017-0050-0005,FINAL ENVIRONMENTAL ASSESSMENT: REDUCING BIRD DAMAGE IN THE STATE OF SOUTH CAROLINA,185,True,
9,US:APHIS-2017-0076-0002,Reducing Bird Conflicts in Idaho: Environmental Assessment,161,True,



--- Canada full text ---
Canada: shape=(32, 5)


,doc_id,title,text_len,has_text,full_text_error
0,CA:https://www.publications.gc.ca/site/eng/9.502842/publication.html,Rapport annuel de ... / Musée canadien de la nature. NM10F-PDF,4595,True,
1,CA:https://www.publications.gc.ca/site/eng/9.506446/publication.html,Annual report / Canadian Museum of Nature. NM10E-PDF,4555,True,
2,CA:https://www.publications.gc.ca/site/eng/9.576782/publication.html,"Report of the Commissioner of the Environment and Sustainable Development to the House of Commons. Chapter 3, : Canadian biodiversity strategy - a follow-up audit FA1-2/2005-3E-PDF",2958,True,
3,CA:https://www.publications.gc.ca/site/eng/9.602385/publication.html,La recherche d'Industrie Canada sur l'investissement étranger : enseignements et incidence sur les politiques / par Ronald Hirshhorn. C21-25/5-1997F-PDF,3100,True,
4,CA:https://www.publications.gc.ca/site/eng/9.689111/publication.html,Evaluation of the Family Violence Initiative - Multiculturalism Program : final report / Submitted by: Positive Outcomes Consulting Services. CH34-11-2002E-PDF,2551,True,
5,CA:https://www.publications.gc.ca/site/eng/9.693537/publication.html,Biodiversity in agriculture : Agriculture and Agri-Food Canada's action plan A42-70/1-1997E-PDF,3120,True,
6,CA:https://www.publications.gc.ca/site/eng/9.693540/publication.html,Biodiversity initiatives : Canadian agricultural producers / by Joyce Greenfield and Nicole Richer. A42-70/3-1997E-PDF,3188,True,
7,CA:https://www.publications.gc.ca/site/eng/9.693864/publication.html,"Canadian biodiversity : ecosystem status and trends 2010 / prepared by federal, provincial and territorial governments. En14-26/2010E-PDF",3159,True,
8,CA:https://www.publications.gc.ca/site/eng/9.699241/publication.html,Canadian biodiversity strategy : Canada's response to the Convention on Biological Diversity En21-134/1995E-PDF,2561,True,
9,CA:https://www.publications.gc.ca/site/eng/9.809198/publication.html,Canada's biodiversity outcomes framework and 2020 goals & targets . CW66-525/2016E-PDF,3205,True,



--- New Zealand full text ---
New Zealand: shape=(7, 5)


,doc_id,title,text_len,has_text,full_text_error
0,NZ:https://www.legislation.govt.nz/act/public/2010/12/en/latest/highlights,,68373,True,
1,NZ:https://www.legislation.govt.nz/act/public/2016/20/en/latest/highlights,,54625,True,
2,NZ:https://www.legislation.govt.nz/act/public/2023/46/en/latest/highlights,,2117779,True,
3,NZ:https://www.legislation.govt.nz/bill/government/2010/130/en/latest/highlights,,90974,True,
4,NZ:https://www.legislation.govt.nz/bill/government/2015/60/en/latest/highlights,,53638,True,
5,NZ:https://www.legislation.govt.nz/bill/government/2022/186/en/latest/highlights,,2171479,True,
6,NZ:https://www.legislation.govt.nz/bill/members/2022/156/en/latest/highlights,,16885,True,


## 7. Build Raw And Cleaned Non-EU Outputs


In [8]:
all_non_eu_rows_df = pd.concat(country_raw_hits.values(), ignore_index=True) if country_raw_hits else empty_raw.copy()
non_eu_raw_hits, non_eu_country_docs = build_non_eu_doc_tables(all_non_eu_rows_df)
non_eu_fulltext_docs = pd.concat(country_fulltext_docs.values(), ignore_index=True) if country_fulltext_docs else empty_full.copy()

if not non_eu_fulltext_docs.empty:
    non_eu_fulltext_docs = non_eu_fulltext_docs.drop_duplicates(subset=["doc_id"]).reset_index(drop=True)
    fulltext_meta = non_eu_fulltext_docs[["doc_id", "title", "date", "year", "retrieval_status"]].rename(
        columns={
            "title": "title_fulltext",
            "date": "date_fulltext",
            "year": "year_fulltext",
            "retrieval_status": "retrieval_status_fulltext",
        }
    )

    non_eu_raw_hits = non_eu_raw_hits.merge(fulltext_meta, on="doc_id", how="left")
    for col in ["title", "date"]:
        full_col = f"{col}_fulltext"
        non_eu_raw_hits[col] = fill_from_fulltext(non_eu_raw_hits[col], non_eu_raw_hits[full_col])
    non_eu_raw_hits["year"] = non_eu_raw_hits["year"].where(non_eu_raw_hits["year"].notna(), non_eu_raw_hits["year_fulltext"])
    if "retrieval_status" not in non_eu_raw_hits.columns:
        non_eu_raw_hits["retrieval_status"] = non_eu_raw_hits["retrieval_status_fulltext"]
    non_eu_raw_hits = non_eu_raw_hits.drop(columns=["title_fulltext", "date_fulltext", "year_fulltext", "retrieval_status_fulltext"])

    non_eu_country_docs = non_eu_country_docs.merge(fulltext_meta, on="doc_id", how="left")
    for col in ["title", "date"]:
        full_col = f"{col}_fulltext"
        non_eu_country_docs[col] = fill_from_fulltext(non_eu_country_docs[col], non_eu_country_docs[full_col])
    non_eu_country_docs["year"] = non_eu_country_docs["year"].where(non_eu_country_docs["year"].notna(), non_eu_country_docs["year_fulltext"])
    if "retrieval_status" not in non_eu_country_docs.columns:
        non_eu_country_docs["retrieval_status"] = non_eu_country_docs["retrieval_status_fulltext"]
    non_eu_country_docs = non_eu_country_docs.drop(columns=["title_fulltext", "date_fulltext", "year_fulltext", "retrieval_status_fulltext"])

non_eu_country_docs = non_eu_country_docs[["doc_id", "country", "jurisdiction", "doc_uid", "source", "title", "url", "lang", "date", "year", "matched_terms", "retrieval_status"]].copy()

output_diag = non_eu_fulltext_docs.groupby("country", as_index=False).agg(
    rows=("doc_id", "count"),
    missing_title=("title", missing_text_count),
    missing_year=("year", lambda s: int(s.isna().sum())),
)
preview_df("output_diag", output_diag, n=len(output_diag))
preview_df("non_eu_raw_hits", non_eu_raw_hits[[c for c in ["doc_id", "country", "source", "title", "year", "url", "matched_terms", "retrieval_status"] if c in non_eu_raw_hits.columns]])
preview_df("non_eu_country_docs", non_eu_country_docs[[c for c in ["doc_id", "country", "source", "title", "year", "url", "matched_terms", "retrieval_status"] if c in non_eu_country_docs.columns]])
preview_df("non_eu_fulltext_docs", non_eu_fulltext_docs[[c for c in ["doc_id", "country", "source", "title", "year", "url", "retrieval_status", "text_len"] if c in non_eu_fulltext_docs.columns]])



NameError: name 'country_raw_hits' is not defined

## 8. Save CSVs

This writes both the pipeline-level outputs and the per-country source outputs so the retrieval layer remains inspectable rather than collapsing immediately into the harmonized corpus.


In [11]:
write_csv(non_eu_raw_hits, OUT_RAW, index=False)
write_csv(non_eu_country_docs, OUT_DOCS, index=False)
write_csv(non_eu_fulltext_docs, OUT_FULLTEXT, index=False)


non_eu_raw_hits: shape=(1130, 14)


,jurisdiction,source,matched_term,term,doc_url,url,title,lang,doc_id,doc_uid,text_url,api_id,api_self,document_id
0,United Kingdom,UK,nature based solutions,nature based solutions,https://www.legislation.gov.uk/wsi/2025/936/contents,https://www.legislation.gov.uk/wsi/2025/936/contents,,en,UK:https://www.legislation.gov.uk/wsi/2025/936/contents,https://www.legislation.gov.uk/wsi/2025/936/contents,NaN,NaN,NaN,NaN
1,United Kingdom,UK,nature based solutions,nature based solutions,https://www.legislation.gov.uk/uksi/2025/726/contents,https://www.legislation.gov.uk/uksi/2025/726/contents,,en,UK:https://www.legislation.gov.uk/uksi/2025/726/contents,https://www.legislation.gov.uk/uksi/2025/726/contents,NaN,NaN,NaN,NaN
2,United Kingdom,UK,nature based solutions,nature based solutions,https://www.legislation.gov.uk/ukpga/2025/5/contents,https://www.legislation.gov.uk/ukpga/2025/5/contents,,en,UK:https://www.legislation.gov.uk/ukpga/2025/5/contents,https://www.legislation.gov.uk/ukpga/2025/5/contents,NaN,NaN,NaN,NaN
3,United Kingdom,UK,nature based solutions,nature based solutions,https://www.legislation.gov.uk/ukpga/1991/56/contents,https://www.legislation.gov.uk/ukpga/1991/56/contents,,en,UK:https://www.legislation.gov.uk/ukpga/1991/56/contents,https://www.legislation.gov.uk/ukpga/1991/56/contents,NaN,NaN,NaN,NaN
4,United Kingdom,UK,nature based solutions,nature based solutions,https://www.legislation.gov.uk/ukpga/2023/10/contents,https://www.legislation.gov.uk/ukpga/2023/10/contents,,en,UK:https://www.legislation.gov.uk/ukpga/2023/10/contents,https://www.legislation.gov.uk/ukpga/2023/10/contents,NaN,NaN,NaN,NaN


non_eu_country_docs: shape=(1001, 7)


,doc_id,jurisdiction,doc_uid,title,url,lang,matched_terms
0,AU:C2004A00485,Australia,https://www.legislation.gov.au/C2004A00485/latest/text,Environment Protection and Biodiversity Conservation Act 1999,https://www.legislation.gov.au/C2004A00485/latest/text,en,"[""nature repair""]"
1,AU:C2007A00175,Australia,https://www.legislation.gov.au/C2007A00175/latest/text,National Greenhouse and Energy Reporting Act 2007,https://www.legislation.gov.au/C2007A00175/latest/text,en,"[""nature repair""]"
2,AU:C2011A00163,Australia,https://www.legislation.gov.au/C2011A00163/latest/text,Clean Energy Regulator Act 2011,https://www.legislation.gov.au/C2011A00163/latest/text,en,"[""nature repair""]"
3,AU:C2023A00121,Australia,https://www.legislation.gov.au/C2023A00121/latest/text,Nature Repair Act 2023,https://www.legislation.gov.au/C2023A00121/latest/text,en,"[""nature repair""]"
4,AU:C2023A00122,Australia,https://www.legislation.gov.au/C2023A00122/asmade/text,Nature Repair (Consequential Amendments) Act 2023,https://www.legislation.gov.au/C2023A00122/asmade/text,en,"[""nature repair""]"


non_eu_fulltext_docs: shape=(1001, 14)


,doc_id,jurisdiction,doc_uid,title,url,lang,source_file,full_text_clean,text_len,has_text,full_text_url,full_text_error,full_text_format,source
0,UK:https://www.legislation.gov.uk/asp/2004/6/contents,United Kingdom,UK:https://www.legislation.gov.uk/asp/2004/6/contents,,https://www.legislation.gov.uk/asp/2004/6/contents,en,https://www.legislation.gov.uk/asp/2004/6/contents,Nature Conservation (Scotland) Act 2004 Skip to main content Skip to navigation legislation.gov.uk https://www.nationalarchives.gov.uk Cymraeg Home Explore our collections Research tools Help and guidance What's new ...,16015,True,https://www.legislation.gov.uk/asp/2004/6/contents,,html,UK
1,UK:https://www.legislation.gov.uk/asp/2018/8/contents,United Kingdom,UK:https://www.legislation.gov.uk/asp/2018/8/contents,,https://www.legislation.gov.uk/asp/2018/8/contents,en,https://www.legislation.gov.uk/asp/2018/8/contents,Forestry and Land Management (Scotland) Act 2018 Skip to main content Skip to navigation legislation.gov.uk https://www.nationalarchives.gov.uk Cymraeg Home Explore our collections Research tools Help and guidance Wh...,11498,True,https://www.legislation.gov.uk/asp/2018/8/contents,,html,UK
2,UK:https://www.legislation.gov.uk/asp/2021/8/contents,United Kingdom,UK:https://www.legislation.gov.uk/asp/2021/8/contents,,https://www.legislation.gov.uk/asp/2021/8/contents,en,https://www.legislation.gov.uk/asp/2021/8/contents,Budget (Scotland) Act 2021 Skip to main content Skip to navigation legislation.gov.uk https://www.nationalarchives.gov.uk Cymraeg Home Explore our collections Research tools Help and guidance What's new About us Sear...,6529,True,https://www.legislation.gov.uk/asp/2021/8/contents,,html,UK
3,UK:https://www.legislation.gov.uk/asp/2022/3/contents,United Kingdom,UK:https://www.legislation.gov.uk/asp/2022/3/contents,,https://www.legislation.gov.uk/asp/2022/3/contents,en,https://www.legislation.gov.uk/asp/2022/3/contents,Budget (Scotland) Act 2022 Skip to main content Skip to navigation legislation.gov.uk https://www.nationalarchives.gov.uk Cymraeg Home Explore our collections Research tools Help and guidance What's new About us Sear...,6529,True,https://www.legislation.gov.uk/asp/2022/3/contents,,html,UK
4,UK:https://www.legislation.gov.uk/asp/2023/2/contents,United Kingdom,UK:https://www.legislation.gov.uk/asp/2023/2/contents,,https://www.legislation.gov.uk/asp/2023/2/contents,en,https://www.legislation.gov.uk/asp/2023/2/contents,Budget (Scotland) Act 2023 Skip to main content Skip to navigation legislation.gov.uk https://www.nationalarchives.gov.uk Cymraeg Home Explore our collections Research tools Help and guidance What's new About us Sear...,6529,True,https://www.legislation.gov.uk/asp/2023/2/contents,,html,UK


## Per-Country Exports


In [12]:
country_raw_paths = build_and_save_country_dfs(non_eu_raw_hits, out_dir=COUNTRY_RAW_DIR, fmt="csv")
print("country_raw_paths =")
display(pd.DataFrame(list(country_raw_paths.items()), columns=["country", "path"]))

COUNTRY_FULLTEXT_DIR.mkdir(parents=True, exist_ok=True)
country_fulltext_paths = {}
for country, df in country_fulltext_docs.items():
    path = COUNTRY_FULLTEXT_DIR / f"nid_policy_{country.lower().replace(' ', '_')}_with_fulltext.csv"
    if SAVE_OUTPUTS:
        df.to_csv(path, index=False)
    country_fulltext_paths[country] = str(path)

print("country_fulltext_paths =")
display(pd.DataFrame(list(country_fulltext_paths.items()), columns=["country", "path"]))


Wrote: c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_non_eu_raw_hits.csv
Shape: (1130, 14)


,jurisdiction,source,matched_term,term,doc_url,url,title,lang,doc_id,doc_uid,text_url,api_id,api_self,document_id
0,United Kingdom,UK,nature based solutions,nature based solutions,https://www.legislation.gov.uk/wsi/2025/936/contents,https://www.legislation.gov.uk/wsi/2025/936/contents,,en,UK:https://www.legislation.gov.uk/wsi/2025/936/contents,https://www.legislation.gov.uk/wsi/2025/936/contents,NaN,NaN,NaN,NaN
1,United Kingdom,UK,nature based solutions,nature based solutions,https://www.legislation.gov.uk/uksi/2025/726/contents,https://www.legislation.gov.uk/uksi/2025/726/contents,,en,UK:https://www.legislation.gov.uk/uksi/2025/726/contents,https://www.legislation.gov.uk/uksi/2025/726/contents,NaN,NaN,NaN,NaN
2,United Kingdom,UK,nature based solutions,nature based solutions,https://www.legislation.gov.uk/ukpga/2025/5/contents,https://www.legislation.gov.uk/ukpga/2025/5/contents,,en,UK:https://www.legislation.gov.uk/ukpga/2025/5/contents,https://www.legislation.gov.uk/ukpga/2025/5/contents,NaN,NaN,NaN,NaN
3,United Kingdom,UK,nature based solutions,nature based solutions,https://www.legislation.gov.uk/ukpga/1991/56/contents,https://www.legislation.gov.uk/ukpga/1991/56/contents,,en,UK:https://www.legislation.gov.uk/ukpga/1991/56/contents,https://www.legislation.gov.uk/ukpga/1991/56/contents,NaN,NaN,NaN,NaN
4,United Kingdom,UK,nature based solutions,nature based solutions,https://www.legislation.gov.uk/ukpga/2023/10/contents,https://www.legislation.gov.uk/ukpga/2023/10/contents,,en,UK:https://www.legislation.gov.uk/ukpga/2023/10/contents,https://www.legislation.gov.uk/ukpga/2023/10/contents,NaN,NaN,NaN,NaN


Wrote: c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_non_eu_country_docs.csv
Shape: (1001, 7)


,doc_id,jurisdiction,doc_uid,title,url,lang,matched_terms
0,AU:C2004A00485,Australia,https://www.legislation.gov.au/C2004A00485/latest/text,Environment Protection and Biodiversity Conservation Act 1999,https://www.legislation.gov.au/C2004A00485/latest/text,en,"[""nature repair""]"
1,AU:C2007A00175,Australia,https://www.legislation.gov.au/C2007A00175/latest/text,National Greenhouse and Energy Reporting Act 2007,https://www.legislation.gov.au/C2007A00175/latest/text,en,"[""nature repair""]"
2,AU:C2011A00163,Australia,https://www.legislation.gov.au/C2011A00163/latest/text,Clean Energy Regulator Act 2011,https://www.legislation.gov.au/C2011A00163/latest/text,en,"[""nature repair""]"
3,AU:C2023A00121,Australia,https://www.legislation.gov.au/C2023A00121/latest/text,Nature Repair Act 2023,https://www.legislation.gov.au/C2023A00121/latest/text,en,"[""nature repair""]"
4,AU:C2023A00122,Australia,https://www.legislation.gov.au/C2023A00122/asmade/text,Nature Repair (Consequential Amendments) Act 2023,https://www.legislation.gov.au/C2023A00122/asmade/text,en,"[""nature repair""]"


Wrote: c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_non_eu_fulltext_docs.csv
Shape: (1001, 14)


,doc_id,jurisdiction,doc_uid,title,url,lang,source_file,full_text_clean,text_len,has_text,full_text_url,full_text_error,full_text_format,source
0,UK:https://www.legislation.gov.uk/asp/2004/6/contents,United Kingdom,UK:https://www.legislation.gov.uk/asp/2004/6/contents,,https://www.legislation.gov.uk/asp/2004/6/contents,en,https://www.legislation.gov.uk/asp/2004/6/contents,Nature Conservation (Scotland) Act 2004 Skip to main content Skip to navigation legislation.gov.uk https://www.nationalarchives.gov.uk Cymraeg Home Explore our collections Research tools Help and guidance What's new ...,16015,True,https://www.legislation.gov.uk/asp/2004/6/contents,,html,UK
1,UK:https://www.legislation.gov.uk/asp/2018/8/contents,United Kingdom,UK:https://www.legislation.gov.uk/asp/2018/8/contents,,https://www.legislation.gov.uk/asp/2018/8/contents,en,https://www.legislation.gov.uk/asp/2018/8/contents,Forestry and Land Management (Scotland) Act 2018 Skip to main content Skip to navigation legislation.gov.uk https://www.nationalarchives.gov.uk Cymraeg Home Explore our collections Research tools Help and guidance Wh...,11498,True,https://www.legislation.gov.uk/asp/2018/8/contents,,html,UK
2,UK:https://www.legislation.gov.uk/asp/2021/8/contents,United Kingdom,UK:https://www.legislation.gov.uk/asp/2021/8/contents,,https://www.legislation.gov.uk/asp/2021/8/contents,en,https://www.legislation.gov.uk/asp/2021/8/contents,Budget (Scotland) Act 2021 Skip to main content Skip to navigation legislation.gov.uk https://www.nationalarchives.gov.uk Cymraeg Home Explore our collections Research tools Help and guidance What's new About us Sear...,6529,True,https://www.legislation.gov.uk/asp/2021/8/contents,,html,UK
3,UK:https://www.legislation.gov.uk/asp/2022/3/contents,United Kingdom,UK:https://www.legislation.gov.uk/asp/2022/3/contents,,https://www.legislation.gov.uk/asp/2022/3/contents,en,https://www.legislation.gov.uk/asp/2022/3/contents,Budget (Scotland) Act 2022 Skip to main content Skip to navigation legislation.gov.uk https://www.nationalarchives.gov.uk Cymraeg Home Explore our collections Research tools Help and guidance What's new About us Sear...,6529,True,https://www.legislation.gov.uk/asp/2022/3/contents,,html,UK
4,UK:https://www.legislation.gov.uk/asp/2023/2/contents,United Kingdom,UK:https://www.legislation.gov.uk/asp/2023/2/contents,,https://www.legislation.gov.uk/asp/2023/2/contents,en,https://www.legislation.gov.uk/asp/2023/2/contents,Budget (Scotland) Act 2023 Skip to main content Skip to navigation legislation.gov.uk https://www.nationalarchives.gov.uk Cymraeg Home Explore our collections Research tools Help and guidance What's new About us Sear...,6529,True,https://www.legislation.gov.uk/asp/2023/2/contents,,html,UK


country_raw_paths =


,country,path
0,Australia,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_country_raw\nid_policy_australia.csv
1,Canada,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_country_raw\nid_policy_canada.csv
2,New Zealand,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_country_raw\nid_policy_new_zealand.csv
3,UK,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_country_raw\nid_policy_uk.csv
4,US,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_country_raw\nid_policy_us.csv


country_fulltext_paths =


,country,path
0,United Kingdom,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_country_fulltext\nid_policy_united_kingdom_with_fulltext.csv
1,Australia,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_country_fulltext\nid_policy_australia_with_fulltext.csv
2,United States,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_country_fulltext\nid_policy_united_states_with_fulltext.csv
3,Canada,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_country_fulltext\nid_policy_canada_with_fulltext.csv
4,New Zealand,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01C_country_fulltext\nid_policy_new_zealand_with_fulltext.csv
